# California Housing — Exploratory Data Analysis

        ## Purpose

        This notebook characterises the **scikit-learn California Housing** dataset before any model is trained. The target is the 1990 census-district median house value, stored in units of $100,000. It is a historical, area-level learning exercise—not a live market predictor or individual property appraisal.

        **Reader contract:** every table and plot is descriptive. A correlation, cluster, or feature pattern is not proof that one variable causes house value.


## Analysis map

        ```text
        fetch maintained source → verify shape and missingness → describe distributions
        → inspect correlations and geography → record modelling implications
        ```

        The dataset is fetched at runtime. This keeps the repository light and makes the data provenance explicit; it also means the first run needs network access or an existing scikit-learn cache.


In [ ]:
# Import only the tools needed for data inspection and static visuals.
        # A consistent theme makes plots legible without changing the evidence.
        from sklearn.datasets import fetch_california_housing
        import matplotlib.pyplot as plt
        import pandas as pd
        import seaborn as sns

        sns.set_theme(style="whitegrid", context="notebook")
        RANDOM_STATE = 42  # Kept for continuity with the modelling notebook.


In [ ]:
# Fetch the maintained dataset as labelled pandas objects. `as_frame=True`
        # preserves feature names, which prevents column-order mistakes in EDA.
        housing = fetch_california_housing(as_frame=True)
        X = housing.data.copy()
        y = housing.target.rename("MedHouseVal")
        housing_frame = pd.concat([X, y], axis=1)

        print(f"Rows: {len(housing_frame):,}")
        print(f"Predictors: {X.shape[1]}")
        print("Target unit: $100,000s")
        housing_frame.head()


## 1. Data-quality boundary

        Before reading relationships, confirm the basic contract: each row is a district, all eight predictors are numeric, and no feature values are missing in the fetched version. The target cap at **5.00001** is a semantic issue rather than a null-value issue—it removes variation at the expensive end and must remain visible in interpretation.


In [ ]:
# Count nulls column by column rather than assuming the public dataset is complete.
        # This result informs whether a model needs an imputation strategy.
        quality = pd.DataFrame({
            "dtype": housing_frame.dtypes.astype(str),
            "missing": housing_frame.isna().sum(),
            "missing_pct": housing_frame.isna().mean().mul(100).round(3),
            "unique_values": housing_frame.nunique(),
        })
        display(quality)
        print(f"Target maximum: {y.max():.5f} ($100,000s) — the documented cap.")


In [ ]:
# Summaries reveal scale and skew. We include 1st/99th percentiles because
        # population and occupancy contain extreme districts that a mean can hide.
        summary = housing_frame.describe(percentiles=[0.01, 0.5, 0.99]).T
        display(summary)


## 2. Target distribution and censoring

        A regression score only has meaning in the context of the target distribution. The histogram and the count at the cap make the dataset's high-end censoring visible. A model cannot recover distinctions the label itself does not contain.


In [ ]:
# The first panel shows the overall target distribution. The second counts
        # districts at the cap, a direct diagnostic for label censoring.
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        sns.histplot(y, bins=45, kde=True, color="#5B3FD6", ax=axes[0])
        axes[0].set(title="District median house value", xlabel="Value ($100,000s)", ylabel="District count")

        cap = y.max()
        cap_count = (y == cap).sum()
        axes[1].bar(["below cap", "at cap"], [len(y) - cap_count, cap_count], color=["#247BA0", "#F4A261"])
        axes[1].set(title="Target censoring diagnostic", ylabel="District count")
        axes[1].text(1, cap_count, f"{cap_count:,}", ha="center", va="bottom", fontweight="bold")
        fig.tight_layout()
        plt.show()


## 3. Feature distributions, associations, and location

        The correlation heatmap is a compact screen for linear association, not a model and not a causal diagram. The scatter uses a sample to avoid painting more than 20,000 points on top of one another. Latitude/longitude is visualised separately because location can encode regional structure that a random row split may leak across the holdout boundary.


In [ ]:
# Spearman correlation is shown alongside Pearson-style visual intuition:
        # it is less sensitive to the very large occupancy and population values.
        correlation = housing_frame.corr(method="spearman")
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(correlation, cmap="vlag", center=0, square=True, ax=ax)
        ax.set_title("Spearman rank correlations (descriptive only)")
        plt.show()


In [ ]:
# A fixed sample makes the plot reproducible while preserving readable density.
        # Income and location are selected because the baseline later relies on them;
        # seeing their structure prevents an importance chart from being over-read.
        plot_frame = housing_frame.sample(n=4_000, random_state=RANDOM_STATE)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
        sns.scatterplot(data=plot_frame, x="MedInc", y="MedHouseVal", alpha=0.22, s=16, color="#5B3FD6", ax=axes[0])
        axes[0].set(title="Income and target", xlabel="Median income (tens of thousands)", ylabel="Value ($100,000s)")
        scatter = axes[1].scatter(plot_frame["Longitude"], plot_frame["Latitude"], c=plot_frame["MedHouseVal"], s=7, alpha=0.45, cmap="viridis")
        axes[1].set(title="Geographic distribution of the target", xlabel="Longitude", ylabel="Latitude")
        fig.colorbar(scatter, ax=axes[1], label="Value ($100,000s)")
        fig.tight_layout()
        plt.show()


## 4. EDA conclusions that constrain modelling

        1. The source has no missing features today, but a training-only imputer remains a safe future-input guard.
        2. Extreme population and occupancy values make robust summaries and residual checks more useful than a single average error.
        3. The target cap limits high-value interpretation.
        4. Coordinates contain strong regional structure, so a random split is only a first baseline. Spatially blocked validation is needed before claiming transfer to unseen areas.

Continue with [`02_model_development.ipynb`](02_model_development.ipynb) for the leakage-safe baseline.
